<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/4_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 下方 fig 排版

### 邏輯闡述

您的需求將依照以下結構層層構建：

1.  **外層容器**：遍歷 `select_pattern_list`，每一個 Pattern 都會產生一個大型的 `html.Div`。
2.  **區隔與標題**：
    * 頂部有一條 **酒紅色橫線**（使用 CSS 的 `borderTop`）。
    * 標題列：左側顯示 Pattern 名稱，右側放置兩個控制按鈕。
3.  **資料過濾與圖片顯示**：
    * 針對該 Pattern 過濾 `pic_df`。
    * **疊圖效果**：利用 CSS 的 `position: relative`（容器）與 `position: absolute`（地圖），將 `map_b64` 精確定位在 `pic_b64` 的左上方。
4.  **網格排版**：
    * 利用 **Flexbox** (`display: flex`, `flex-wrap: wrap`) 讓每兩個圖片組（2 units per row）自動換行，每個圖片組寬度設為約 `50%`。


In [ ]:
from dash import html

def create_layout(select_pattern_list, pic_df):
    main_layout = []

    # 取得清單長度，用來判斷是否為最後一個 pattern
    total_patterns = len(select_pattern_list)

    for i, pattern in enumerate(select_pattern_list):
        # 根據 pattern 篩選資料
        df_filter = pic_df[pic_df['CLASS_TYPE'] == pattern]

        # 建立區塊內容
        pattern_section = [
            # 1. 頂部酒紅色橫線
            html.Hr(style={'borderTop': '3px solid #800000', 'margin': '20px 0'}),

            # 2. 標題與按鈕
            html.Div([
                html.H3(f"Pattern: {pattern}", style={'display': 'inline-block', 'margin': '0'}),
                html.Div([
                    html.Button("Hide Map", id={'type': 'btn-hide-map', 'index': pattern}, n_clicks=0, style={'marginRight': '5px'}),
                    html.Button("Hide Center", id={'type': 'btn-hide-center', 'index': pattern}, n_clicks=0),
                ], style={'float': 'right'})
            ], style={'marginBottom': '15px', 'overflow': 'hidden'}),

            # 3 & 4. 圖片網格容器
            html.Div([
                html.Div([
                    # 圖片組容器 (relative)
                    html.Div([
                        # A. 主圖 (pic_b64)
                        html.Img(
                            src=f"data:image/png;base64,{row['pic_b64']}",
                            style={'width': '100%', 'display': 'block'}
                        ),

                        # B. 地圖 (map_b64) - 左上方
                        html.Img(
                            id={'type': 'img-map', 'index': f"{pattern}-{i}"},
                            src=f"data:image/png;base64,{row['map_b64']}",
                            style={
                                'position': 'absolute',
                                'top': '5px',
                                'left': '5px',
                                'width': '25%',
                                'border': '1px solid white',
                                'boxShadow': '2px 2px 5px rgba(0,0,0,0.5)',
                                'zIndex': '10' # 確保在最上層
                            }
                        ),

                        # C. 新增：正中心圖片 (img2)
                        html.Img(
                            id={'type': 'img-center', 'index': f"{pattern}-{i}"},
                            src=f"data:image/png;base64,{row['img2_b64']}", # 假設欄位名為 img2_b64
                            style={
                                'position': 'absolute',
                                'top': '50%',      # 移至父容器高度 50% 處
                                'left': '50%',     # 移至父容器寬度 50% 處
                                'transform': 'translate(-50%, -50%)', # 往回位移自身寬高的一半，達成完美置中
                                'width': '20%',    # 根據需求調整大小
                                'zIndex': '10'
                            }
                        )
                    ], style={
                        'position': 'relative',
                        'border': '2px solid #ddd',
                        'padding': '5px',
                        'margin': '5px'
                    })
                ], style={'width': '48%', 'display': 'inline-block'})
                for _, row in df_filter.iterrows()
            ], style={'display': 'flex', 'flexWrap': 'wrap', 'justifyContent': 'space-between'}),
        ]

        main_layout.append(html.Div(pattern_section))

    # --- 重點：迴圈跑完後，直接在最後補上一條線 ---
    main_layout.append(html.Hr(style={'borderTop': '3px solid #800000', 'margin': '20px 0'}))

    return html.Div(main_layout)

### 關鍵技術說明：

* **圖片重疊 (Overlay)**：在外部 `Div` 使用 `position: relative`，內部的地圖 `Img` 使用 `position: absolute`。這樣 `top: 5px` 和 `left: 5px` 就會相對於該圖片組的左上角。
* **每列兩個 (Two-per-row)**：外層容器使用 `display: flex` 與 `flex-wrap: wrap`。內部圖片組設定 `width: 48%`（留一點空間給 margin），這會讓 Dash 自動在第三個物件時進行換行。
* **動態 ID**：在按鈕處我使用了 Pattern 作為 `index` 的 **Pattern-matching Callback** 格式（例如 `{'type': 'btn-hide-map', 'index': pattern}`），這對於之後您要撰寫按鈕功能會非常有幫助。

## 2. 圖片堆疊

在 CSS 中，當你設定一個元素為 `position: absolute`（絕對定位）時，它會脫離正常的排版流（Document Flow），並開始尋找它的 **「參考基準點」**。



### 2-1. ▼為什麼要有 `relative`？

簡單來說：**`absolute` 永遠在找離它最近的「有定位（非 static）」的父元素。**

1.  **預設情況（Static）**：
    所有的 `html.Div` 預設都是 `static`。如果你沒有在父層設定 `relative`，那麼內部的 `absolute` 地圖會一直往上找，直到找到 `body`（網頁的最外層）。這會導致你的地圖全部疊在網頁的左上角，而不是圖片的左上角。

2.  **設定 `relative` 的作用**：
    當你在外部 `Div` 設定了 `position: relative`，你就像是在這個 `Div` 釘上了一個 **「座標原點」**。
    * 這告訴內部的子元素：「我就在這裡，你可以以我為準。」
    * 內部的 `Img` 設定 `absolute` 後，發現父層有 `relative`，它就會停止往上找，並把父層的左上角定義為 $(0, 0)$。

---

### 圖解定位邏輯

* **外部 Div (`relative`)**：建立一個邊界框。
* **內部地圖 (`absolute`)**：
    * `top: 5px`：距離父層頂部邊界 5px。
    * `left: 5px`：距離父層左側邊界 5px。

### 屬性對比表

| 屬性值 | 行為特性 | 對子元素的影響 |
| :--- | :--- | :--- |
| **`static`** (預設) | 照著順序排隊，不提供座標參考。 | 子元素 `absolute` 會無視它，繼續往外找。 |
| **`relative`** | **留在原地排隊**，但開啟座標參考功能。 | 子元素 `absolute` 會以它的左上角為起點。 |
| **`absolute`** | **脫離排隊**，漂浮在空中。 | 根據最近的 `relative`/`absolute` 祖先來定位。 |

這就是為什麼在 Dash 或 Web 開發中，處理「圖上疊圖」或是「圖片加標籤」時，**父 relative / 子 absolute** 是最標準的公式。

您的資料分析工具中，每個 `wafer_id` 的圖片組都是獨立的 `relative` 容器，這樣每一張地圖才都能乖乖待在屬於自己的主圖左上角，不會亂跑。

### 2-2. ▼Center 框 重點說明：

1.  **正中心定位技巧**：
    * `top: 50%` 和 `left: 50%` 會將圖片的「左上角」放在中心點。
    * `transform: translate(-50%, -50%)` 是最關鍵的步驟，它會讓圖片根據自身的寬高向左、向上平移一半，這樣圖片的「中心點」才會精確對準容器的「中心點」。

2.  **zIndex**：
    * 我額外加上了 `zIndex: 10`，確保這兩個絕對定位的圖片（地圖和中心圖）不會被主圖遮擋，並且可以疊在最上層。

## 3. callback

In [ ]:
from dash import Input, Output, State, ALL, ctx

# --- Callback 1: 控制 Map 顯示與按鈕文字 ---
@app.callback(
    Output({'type': 'btn-hide-map', 'index': ALL}, 'children'),
    Output({'type': 'img-map', 'index': ALL}, 'style'),
    Input({'type': 'btn-hide-map', 'index': ALL}, 'n_clicks'),
    State({'type': 'img-map', 'index': ALL}, 'style'),
    prevent_initial_call=True
)
def toggle_all_maps(n_clicks_list, current_styles):
    # 只要清單中任一個按鈕被點擊 (總次數變動)
    total_clicks = sum(n_clicks_list) if n_clicks_list else 0

    # 偶數次點擊 = 顯示 (Hide Map), 奇數次點擊 = 隱藏 (Show Map)
    is_hidden = (total_clicks % 2 == 1)

    new_label = "Show Map" if is_hidden else "Hide Map"
    display_val = "none" if is_hidden else "block"

    # 更新所有按鈕文字
    btn_labels = [new_label for _ in n_clicks_list]

    # 更新所有圖片 Style (保留原本的 absolute 定位，僅改 display)
    new_styles = []
    for s in current_styles:
        s['display'] = display_val
        new_styles.append(s)

    return btn_labels, new_styles


# --- Callback 2: 控制 Center (img2) 顯示與按鈕文字 ---
@app.callback(
    Output({'type': 'btn-hide-center', 'index': ALL}, 'children'),
    Output({'type': 'img-center', 'index': ALL}, 'style'),
    Input({'type': 'btn-hide-center', 'index': ALL}, 'n_clicks'),
    State({'type': 'img-center', 'index': ALL}, 'style'),
    prevent_initial_call=True
)
def toggle_all_centers(n_clicks_list, current_styles):
    total_clicks = sum(n_clicks_list) if n_clicks_list else 0
    is_hidden = (total_clicks % 2 == 1)

    new_label = "Show Center" if is_hidden else "Hide Center"
    display_val = "none" if is_hidden else "block"

    btn_labels = [new_label for _ in n_clicks_list]
    new_styles = []
    for s in current_styles:
        s['display'] = display_val
        new_styles.append(s)

    return btn_labels, new_styles

### 3-1. 重點解析：

1.  **`ALL` 的威力**：
    * `Input({'type': 'btn-hide-map', 'index': ALL}, 'n_clicks')` 會傳入一個整數清單。
    * 當你按下頁面上「任何一個」Hide Map 按鈕，該按鈕的 `n_clicks` 會加 1，導致整個清單的總和改變，進而觸發 Callback。
2.  **狀態切換 (Toggle Logic)**：
    * 我們利用 `total_clicks % 2` 來判斷當前應該是「顯示」還是「隱藏」。這確保了同步性：按一下任一鈕全部消失，再按一下任一鈕全部出現。
3.  **Style 的保留**：
    * 圖片的定位（`absolute`, `top`, `left`）都存在 `style` 字典裡。我們透過 `State` 抓取當前的 `current_styles`，只修改 `display` 鍵值，這樣才不會導致圖片在顯示時失去原本的疊加位置。
4.  **`ctx` (Callback Context)**：
    * 如果你未來需要知道到底是「哪一個」按鈕被按，可以使用 `dash.callback_context` (或 `ctx`)，但以你目前「全域連動」的需求，直接算總次數是最穩健的做法。